# Resumo Detalhado dos Dados PAM (Produção Agrícola Municipal)

Este notebook consolida o perfil dos dados da PAM, analisando variáveis críticas como área plantada, colhida e valor da produção por município.

In [11]:
import os
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import numpy as np

# Configuração de caminhos
# O notebook está em notebooks/etl-dados-necessarios/pam/data/resume/
# O dado está em notebooks/etl-dados-necessarios/pam/data/bronze/pam/
BASE_DIR = Path.cwd().parent.parent # notebooks/etl-dados-necessarios/pam/
PAM_BRONZE_DIR = BASE_DIR / "data/bronze/pam"
OUTPUT_DIR = BASE_DIR / "data/resume"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def analyze_pam_chunks(directory):
    files = list(directory.rglob("*.parquet"))
    if not files:
        print(f"❌ Nenhum arquivo .parquet encontrado em {directory}.")
        return None
    
    print(f"🔍 Analisando {len(files)} chunks da PAM...")
    
    # Lendo o primeiro arquivo para pegar a estrutura
    sample_df = pd.read_parquet(files[0])
    cols = sample_df.columns
    num_rows_total = 0
    
    # Dicionário para acumular métricas
    col_summary = {col: {"null_count": 0, "unique_values": set(), "min": None, "max": None, "dtype": str(sample_df[col].dtype)} for col in cols}
    
    for file in tqdm(files, desc="Processando Chunks"):
        df = pd.read_parquet(file)
        num_rows_total += len(df)
        
        for col in cols:
            series = df[col]
            col_summary[col]["null_count"] += int(series.isna().sum())
            
            # Para PAM, os valores únicos de categorias (Cultura, Município, Ano) são cruciais
            if len(col_summary[col]["unique_values"]) < 1000: # Limite para não estourar memória
                # Garante que adicionamos como string para evitar problemas de tipos mistos no set
                col_summary[col]["unique_values"].update(series.dropna().unique().astype(str).tolist())
            
            # Min/Max para numéricos e datas
            # Usando pd.api.types para evitar erro com StringDtype no numpy
            if pd.api.types.is_numeric_dtype(series.dtype):
                c_min = series.min()
                c_max = series.max()
                if col_summary[col]["min"] is None or (c_min is not None and c_min < col_summary[col]["min"]):
                    col_summary[col]["min"] = c_min
                if col_summary[col]["max"] is None or (c_max is not None and c_max > col_summary[col]["max"]):
                    col_summary[col]["max"] = c_max

    # Consolidando resultados
    final_summary = []
    for col, s in col_summary.items():
        final_summary.append({
            "column": col,
            "dtype": s["dtype"],
            "total_rows": num_rows_total,
            "null_count": s["null_count"],
            "null_percent": round((s["null_count"] / num_rows_total) * 100, 2) if num_rows_total > 0 else 0,
            "unique_values_count": len(s["unique_values"]),
            "min": s["min"],
            "max": s["max"]
        })
    
    return pd.DataFrame(final_summary)

summary_df = analyze_pam_chunks(PAM_BRONZE_DIR)

if summary_df is not None:
    output_file = OUTPUT_DIR / "resumo_detalhado_pam.csv"
    summary_df.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"\n✅ Resumo consolidado exportado para: {output_file}")
    display(summary_df)


🔍 Analisando 5 chunks da PAM...


Processando Chunks: 100%|██████████| 5/5 [00:00<00:00, 40.24it/s]


✅ Resumo consolidado exportado para: /home/vinicius/Downloads/estudo/fatec/SABADO-TE-ANALISE-DADOS/notebooks/etl-dados-necessarios/pam/data/resume/resumo_detalhado_pam.csv


,column,dtype,total_rows,null_count,null_percent,unique_values_count,min,max
0,NC,str,444170,0,0.0,2,None,None
1,NN,str,444170,0,0.0,2,None,None
2,MC,str,444170,0,0.0,6,None,None
3,MN,str,444170,0,0.0,6,None,None
4,V,str,444170,0,0.0,11092,None,None
5,D1N,str,444170,0,0.0,5564,None,None
6,D2C,str,444170,0,0.0,11,None,None
7,D2N,str,444170,0,0.0,11,None,None
8,D3C,str,444170,0,0.0,6,None,None
9,D3N,str,444170,0,0.0,6,None,None


### Análise de Consistência (Série Temporal)
Diferente de dados de fiscalização, a PAM é uma série histórica. Vamos verificar se temos dados para todos os anos esperados e quais culturas são predominantes.

In [12]:
# Carregando uma amostra maior para análise de negócio
files = list(PAM_BRONZE_DIR.rglob("*.parquet"))
if files:
    # Lendo os 2 primeiros chunks
    df_sample = pd.concat([pd.read_parquet(f) for f in files[:2]])
    
    # IMPORTANTE: No estágio BRONZE da PAM, a linha 0 costuma conter nomes amigáveis (metadados),
    # o que força todas as colunas a serem lidas como strings/object.
    df_clean = df_sample.copy()
    
    # 1. Remover linhas de cabeçalho repetidas (metadados)
    df_clean = df_clean[df_clean['V'] != 'Valor']
    
    # 2. Tentar converter a coluna de Valor (V) para numérico
    df_clean['V'] = pd.to_numeric(df_clean['V'], errors='coerce')

    print("--- TOP 5 PRODUTOS NO SAMPLE ---")
    culture_col = [c for c in df_clean.columns if 'D4N' in c or 'Produto' in c]
    if culture_col:
        # Exportando Top Culturas para CSV
        top_culturas = df_clean[df_clean[culture_col[0]] != 'Produto (Nome)'][culture_col[0]].value_counts().head(10).reset_index()
        top_culturas.columns = ['Produto', 'Contagem']
        
        output_top = OUTPUT_DIR / "resumo_top_culturas_pam.csv"
        top_culturas.to_csv(output_top, index=False, encoding="utf-8-sig")
        print(f"📊 Top culturas exportado para: {output_top}")
        display(top_culturas.head(5))
    
    print("\n--- ESTATÍSTICAS DE PRODUÇÃO (Coluna V) ---")
    numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
    
    if len(numeric_cols) > 0:
        stats_df = df_clean[numeric_cols].describe()
        
        # Exportando Estatísticas para CSV
        output_stats = OUTPUT_DIR / "resumo_estatistico_producao_pam.csv"
        stats_df.to_csv(output_stats, encoding="utf-8-sig")
        print(f"📈 Estatísticas de produção exportadas para: {output_stats}")
        display(stats_df)
    else:
        print("⚠️ Nenhuma coluna numérica identificada. Verifique se a conversão de 'V' funcionou.")
        print(f"Dtypes atuais:\n{df_clean.dtypes}")

--- TOP 5 PRODUTOS NO SAMPLE ---
📊 Top culturas exportado para: /home/vinicius/Downloads/estudo/fatec/SABADO-TE-ANALISE-DADOS/notebooks/etl-dados-necessarios/pam/data/resume/resumo_top_culturas_pam.csv


,Produto,Contagem
0,Total,177664



--- ESTATÍSTICAS DE PRODUÇÃO (Coluna V) ---
📈 Estatísticas de produção exportadas para: /home/vinicius/Downloads/estudo/fatec/SABADO-TE-ANALISE-DADOS/notebooks/etl-dados-necessarios/pam/data/resume/resumo_estatistico_producao_pam.csv


,V
count,1.258820e+05
mean,1.309307e+04
std,1.099284e+05
min,1.000000e+00
25%,1.000000e+02
50%,1.000000e+02
75%,1.240000e+03
max,8.313733e+06
